In [47]:

from dataclasses import dataclass, field
from typing import List, Dict
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter
# from sentence_transformers import SentenceTransformer
import json
import numpy as np
from openai import AsyncOpenAI
import pandas
import difflib
import re


In [63]:

# A provider-agnostic description of this tool, in JSON-Schema-ish shape.
# Translate this into whatever your chosen model's API actually expects
# (OpenAI/Anthropic tool schemas differ slightly in envelope, not in spirit).
CALCULATE_NOI_TOOL_SCHEMA = {
    "type": "function",
    "function": {
        "name": "calculate_noi",
        "description": (
            "Compute Net Operating Income (revenue - operating expenses) for a "
            "property and period. Always use this tool for NOI questions -- "
            "never estimate or compute NOI yourself."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "property_id": {"type": "integer", "description": "The property's numeric id"},
                "period": {"type": "string", "description": "e.g. 'Q3-2026'"},
            },
            "required": ["property_id", "period"],
        },
    }
}


FIND_PROPERTY_TOOL_SCHEMA = {
    "type": "function",
    "function": {
        "name": "find_property",
        "description": "Fuzzy search for a property by name, returning the top matches and their IDs.",
        "parameters": {
            "type": "object",
            "properties": {
                "name_query": {"type": "string", "description": "The Property name that the user is asking about"},
                "csv_path": {"type": "string", "description": "the path of the properties csv file"},
            },
            "required": ["name_query"],
        }
    }
}

NORMALISE_PERIOD_TOOL_SCHEMA = {
    "type": "function",
    "function": {
        "name": "normalize_period",
        "description": "Normalizes the format of a period string to a consistent format (e.g. '2026 q3' -> 'Q3-2026').",
        "parameters": {
            "type": "object",
            "properties": {
                "raw_period": {"type": "string", "description": "The period string to normalize."},
            },
            "required": ["raw_period"],
        }
    }
}


tools = [CALCULATE_NOI_TOOL_SCHEMA, FIND_PROPERTY_TOOL_SCHEMA, NORMALISE_PERIOD_TOOL_SCHEMA]


In [64]:
client = AsyncOpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

In [5]:
@dataclass
class Chunk:
    doc_id: str
    text: str
    acl: List[str]
    embedding: List[float] = field(default_factory=list)
    score: float = 0.0


In [6]:
acl = open('documents/manifest.json')
acl_json = json.loads(acl.read())

In [7]:
async def load_documents(
    documents_dir: str = "documents",
) -> List[Chunk]:
    directory = Path(documents_dir)
    files = sorted(directory.glob("*.txt"))

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=300,
        chunk_overlap=30,
        separators=["\n\n", "\n", " "],
    )

    chunks = []

    for doc_id, file in enumerate(files):
        acl_file = acl_json[file.name]

        text = file.read_text(encoding="utf-8")
        texts = splitter.split_text(text)

        for chunk_text in texts:
            chunks.append(
                Chunk(
                    doc_id=doc_id,
                    text=chunk_text,
                    embedding=[], 
                    acl=acl_file["acl"],
                )
            )

    if not chunks:
        return []

    response = await client.embeddings.create(
        model="nomic-embed-text",
        input=[chunk.text for chunk in chunks],
    )

    for item in response.data:
        chunks[item.index].embedding = item.embedding

    return chunks

In [8]:
chunks = await load_documents('documents')

chunks

[Chunk(doc_id=0, text='Q3 2026 financial summary for Maple Ridge Apartments: total revenue $412,000, operating expenses $158,000.', acl=['finance'], embedding=[-0.023673515766859055, 0.06083952635526657, -0.18733640015125275, -0.016874263063073158, -0.021852828562259674, -0.0849725753068924, 0.04713159799575806, -0.0321686752140522, 0.02878238447010517, -0.0028591479640454054, -0.01816963590681553, 0.020062115043401718, 0.024425113573670387, -0.02642214484512806, 0.07171696424484253, -0.006751430686563253, -0.008451433852314949, -0.03771325200796127, 0.011736211366951466, 0.038377292454242706, 0.02693057619035244, 0.02223809063434601, -0.03365841880440712, -0.00526709808036685, 0.0800999104976654, 0.048221372067928314, -0.00878681056201458, 0.026253819465637207, 0.024320069700479507, -0.04890434071421623, 0.011167236603796482, 0.002031774027273059, -0.026347311213612556, -0.027912035584449768, -0.11236124485731125, -0.0004586331488098949, 0.005749830044806004, -0.001831277273595333, 0.

In [23]:
def is_authorized(chunk_acl: List[str], user_groups: List[str]) -> bool:
    """A chunk is authorized for a user if 'all' is in its acl, or if any
    of the user's groups intersects the chunk's acl.
    """
    return ('all' in chunk_acl or any(group in chunk_acl for group in user_groups)) and user_groups != ['']

is_authorized(['all'], ['finance'])

True

In [60]:
async def retrieve(query: str, user_groups: List[str], top_k: int = 3) -> List[Chunk]:
    """Return only chunks the given user_groups are authorized to see,
    ranked by embedding similarity to `query`.
    """
    query_embeddings = await client.embeddings.create(
        model="nomic-embed-text",
        input=query,
    )
    
    query_vec = np.array(query_embeddings.data[0].embedding)
    query_vec = query_vec / np.linalg.norm(query_vec)
    
    scored_chunks = []
    for chunk in chunks:
        if not is_authorized(chunk.acl, user_groups):
            continue

        chunk_vec = np.array(chunk.embedding)
        chunk_vec = chunk_vec / np.linalg.norm(chunk_vec)

        sim = float(query_vec @ chunk_vec)  # cosine similarity
        chunk.score = sim
        scored_chunks.append(chunk)

    scored_chunks.sort(key=lambda x: x.score, reverse=True)
    return [chunk for chunk in scored_chunks[:top_k]]

In [61]:
docs = await retrieve(
    'What is the security deposit amount for the Oak Hill Plaza Unit 12 lease renewal',
    ['test']
)

C:\Users\steve\AppData\Local\Temp\ipykernel_26064\1421546676.py:1: RuntimeWarning: coroutine 'retrieve' was never awaited
  docs = await retrieve(


In [12]:
docs

[Chunk(doc_id=1, text='Company holiday schedule and office policies for 2026. The office is closed on all federal holidays and the last week of December.', acl=['all'], embedding=[-0.006890485063195229, 0.02848479151725769, -0.16603171825408936, -7.143559923861176e-05, -0.005832736846059561, 0.0040039015002548695, -0.001050761784426868, -0.030635224655270576, 0.09643089771270752, -0.05045932158827782, -0.022721407935023308, 0.027377232909202576, 0.055657804012298584, -0.012307912111282349, 0.021220089867711067, 0.03414827585220337, 0.037825457751750946, -0.031143097206950188, -0.026738110929727554, 0.04837318882346153, 0.05223453789949417, 0.03706425428390503, -0.015318496152758598, -0.024762270972132683, 0.04734046757221222, 0.08899948000907898, 0.04546631872653961, 0.044457145035266876, -0.004793550819158554, -0.023733744397759438, -0.03624742478132248, 0.0242727342993021, -0.014587529934942722, -0.03613310679793358, 0.001443024491891265, -0.03568746894598007, -0.007719832472503185, 

In [44]:
def find_property(name_query: str, csv_path: str = "data/properties.csv") -> list:
    """Fuzzy search for a property by name, returning the top matches and their IDs."""
    properties = pandas.read_csv(csv_path)
    all_names = properties['name'].tolist()
    
    closest_matches = difflib.get_close_matches(
        word=name_query, 
        possibilities=all_names, 
        n=3, 
        cutoff=0.5 
    )
    
    if not closest_matches:
        return {"error": "No properties found matching that name."}

    results = []
    for match in closest_matches:
        row = properties[properties['name'] == match].iloc[0]
        results.append({
            "property_id": int(row['property_id']),
            "name": row['name']
        })
        
    return {"matches": results}

In [48]:
def normalize_period(raw_period: str) -> str:
    """Takes formats like '2026 q3' or 'Q3 2026' and returns 'Q3-2026'"""
    s = str(raw_period).upper().replace(" ", "")
    if m := re.fullmatch(r"Q([1-4])[-/]?(\d{4})", s):  
        return f"Q{m[1]}-{m[2]}"
    elif m := re.fullmatch(r"(\d{4})[-/]?Q([1-4])", s):  
        return f"Q{m[2]}-{m[1]}"
    return raw_period # Return as-is if we can't parse it


In [50]:
period = normalize_period('2026 q3')
period

'Q3-2026'

In [46]:
matches_properties = find_property('Mapl Ridgeppartments')
matches_properties

{'matches': [{'property_id': 1, 'name': 'Maple Ridge Apartments'}]}

In [54]:

def calculate_noi(property_id: int, period: str, csv_path: str = "data/properties.csv") -> float:
    """Return revenue - operating_expenses for the given property_id and
    period, read directly from data/properties.csv. No model involved.
    """

    properties = pandas.read_csv(csv_path)
    row = properties.loc[(properties['property_id'] == property_id) & (properties['period'] == period)]

    return float(row.iloc[0]["revenue"] - row.iloc[0]["operating_expenses"])


In [55]:
if len(matches_properties['matches']) > 0:
    noi = calculate_noi(property_id=matches_properties['matches'][0]['property_id'], period=period)
    print(noi)

254000.0


In [84]:
SYSTEM_PROMPT = """
You are a retrieval-grounded assistant for property and financial questions.

You answer using two sources:
1. Authorized context chunks, provided in the user's message under "CONTEXT".
2. Tools, which you call to look up IDs, normalize periods, and calculate NOI.

## Grounding rules

1. Context chunks are your only source of facts. Answer ONLY from information explicitly stated in them or returned by a tool.
2. Do NOT use general or world knowledge to fill gaps. If the context does not contain the answer, say what is missing.
3. Do NOT invent or assume facts, numbers, dates, names, property details, financial values, or relationships between entities.
4. Use the most relevant chunks; you do not need to use all of them.
5. Combine chunks only when they are consistent. If chunks conflict, do not pick one silently: state the conflict and cite the chunks that support each side.

## Citations

- Each chunk in CONTEXT has an id, shown as [chunk_id].
- Cite every fact taken from the context with its id, for example: "Revenue was $120,000 [3-1]."
- Use only ids that appear in CONTEXT. Never invent an id.
- Facts that come from a tool result are cited as (tool: tool_name), not as a chunk.

## Available tools

- `find_property(name_query)`: fuzzy-matches a property name and returns candidate property IDs.
- `normalize_period(raw_period)`: converts a period such as "2026 q3" or "third quarter 2026" into the standard format, e.g. "Q3-2026".
- `calculate_noi(property_id, period)`: returns the Net Operating Income for a property and period.

## When to use tools

- Use `find_property` when you need a property ID and the user gave only a name. Do not guess IDs.
- Use `normalize_period` when the user's period is not already in the form "Q3-2026". Always pass its output to `calculate_noi`.
- Use `calculate_noi` for ANY question that asks for NOI. Never compute NOI yourself from revenue and expenses, never do the arithmetic, and never state an NOI value before the tool returns it. Do not modify the returned value.
- Do NOT use tools for questions that are not about NOI (for example revenue, expenses, occupancy, lease terms). Answer those from the context chunks. A property ID or period from the chunks is only needed if a later NOI call requires it.

## Calling tools in order

- Some calls depend on others: `calculate_noi` needs the `property_id` from `find_property` and the period from `normalize_period`. Wait for those results before calling it.
- Independent calls, such as `find_property` and `normalize_period`, may be made together in the same turn.
- You can make as many tool calls as you need, one round at a time, until you have everything required.
- Do not repeat a call with the same arguments. Reuse the earlier result.
- If a tool returns an error, read the message, fix the arguments, and retry once. If it still fails, tell the user what went wrong.

## Handling ambiguity and missing information

- If `find_property` returns several plausible matches and the user's name does not clearly identify one, ask the user which property they mean. Do not choose for them.
- If `find_property` returns no match, say so.
- If the property or period cannot be determined from the question, the context, or the tools, ask for the missing information instead of guessing.

## Final answer

- Be concise and answer the question directly.
- State the property name, period, and value so the answer is unambiguous.
- Include citations as described above.
- Do not describe your internal reasoning or the tool-calling steps unless the user asks.

## Example

User: "What is the NOI for Maple Ridge Apartments in 2026 q3?"
Steps: call `find_property("Maple Ridge Apartments")` and `normalize_period("2026 q3")`, then call `calculate_noi` with the returned property_id and "Q3-2026", then answer with the tool's value.

User: "What was Maple Ridge Apartments' total revenue in Q3 2026?"
Steps: no NOI is requested, so answer from the context chunks with citations. If the chunks do not contain it, say the available context does not include that figure.
"""

GROUNDING_SYSTEM_PROMPT = """
You are a grounding verifier for a retrieval-augmented generation system.

Your task is to determine whether the ANSWER is fully supported by the
provided CONTEXT.

Rules:

1. Evaluate only whether the claims in the ANSWER are supported by the
   CONTEXT.

2. Do not use your own knowledge or outside information.

3. Every factual claim in the ANSWER must be supported by the CONTEXT.

4. If the ANSWER contains a factual claim that is not supported by the
   CONTEXT, the answer is not grounded.

5. Paraphrasing is allowed. The answer does not need to use the exact
   wording from the context.

6. Reasonable mathematical transformations are allowed when they are
   directly supported by the context.

7. If the answer contains no unsupported factual claims, return grounded=true.

Return ONLY valid JSON in this format:

{
    "grounded": true,
    "unsupported_claims": []
}

or:

{
    "grounded": false,
    "unsupported_claims": [
        "The unsupported claim here."
    ]
}
"""

In [75]:
TOOL_REGISTRY = {
    "calculate_noi": calculate_noi,
    "find_property": find_property,
    "normalize_period": normalize_period,
}

In [ ]:



async def call_model(prompt: str, tools: list | None = None) -> Dict:
    """Call your chosen real language model.

    Return a normalized dict shaped like:
        {"text": str | None, "tool_call": {"name": str, "arguments": dict} | None}

    - If the model just answers, `text` is set and `tool_call` is None.
    - If the model wants to call a tool (only possible when `tools` is
      passed), `tool_call` is set and `text` may be None.

    You will need to translate this normalized shape to/from whatever your
    provider's actual API expects (OpenAI, Anthropic, and Ollama all differ
    slightly in envelope). That translation is the point of this task.
    """ 

    messages = [
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT,
                },
                {
                    "role": "user",
                    "content": prompt
                },
            ]
    executed = []

    while True:
        completion = await client.chat.completions.create(
            model="glm-4.7-flash:latest",
            messages=messages,
            tools=tools,
        )

        message = completion.choices[0].message
        print(message)
        if not message.tool_calls:
            answer = message.content

            return {
                "text": answer,
                "tool_call": executed[-1] if executed else None,
                "tool_calls": executed
            }

        tool_calls = message.tool_calls


        messages.append(
            {
                "role": "tool",
                "tool_calls": [
                    {
                        "id": tc.id,
                        "type": "function",
                        "function": {
                            "name": tc.function.name,
                            "arguments": tc.function.arguments,
                        },
                    }
                    for tc in message.tool_calls
                ],
                "content": message.content or "",
            }
        )
        
        for tool_call in tool_calls:
            arguments = json.loads(tool_call.function.arguments)
            name = tool_call.function.name

            func = TOOL_REGISTRY.get(name)
            if func is None:
                raise ValueError(f"Unknown tool: {name}")
            result = func(**arguments)

            executed.append({"name": name, "arguments": arguments})

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps({"result": result}),
            })

    return {
        "text": "Stopped: too many tool-calling steps.",
        "tool_call": executed[-1] if executed else None,
        "tool_calls": executed,
    }

In [ ]:
def is_grounded(answer: str, chunks: List[Chunk]) -> bool:
    """Task 1 grounding check: verify the answer's key claims (names,
    numbers, dates) actually appear in the retrieved chunks it was
    supposedly based on. A simple substring/entity check is enough -- it
    needs to exist, not be sophisticated.
    """

    messages = [
        {
            "role": "system",
            "content": GROUNDING_SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": f"""
                    CONTEXT:
                        <context>
                            {chunks}
                        </context>

                    ANSWER:
                        <answer>
                            {answer}
                        </answer>
                """,
            },
        ]

    completion = await client.chat.completions.create(
        model="glm-4.7-flash:latest",
        messages=messages,
        temperature=0,
    )

    content = completion.choices[0].message.content

    try:
        result = json.loads(content)
    except json.JSONDecodeError:
        return False

    return result.get("grounded", False) is True 

In [ ]:
def answer_question(query: str, user_groups: List[str]) -> Dict:
    """Orchestration for both tasks:

    - Retrieve authorized context for the query via retrieve().
    - Call the model WITH the calculate_noi tool available
      (CALCULATE_NOI_TOOL_SCHEMA) so the model itself decides whether this
      question needs the tool -- do not pre-classify the query yourself
      with keywords/regex and route around the model.
    - If the model requests the tool call, execute calculate_noi() in code,
      send the result back to the model, and get its final narrated answer.
      The model must never be allowed to just state the NOI number itself.
    - If the model answers directly instead, apply is_grounded() to check
      it against the retrieved chunks.

    Return a dict shaped like:
        {"answer": str, "citations": [...], "grounded": bool, "used_tool": bool}
    """

    chunks = await retrieve(query, user_groups)
    prompt = f" Authorized context: {chunks} \n User question:{query}"
    response = call_model(prompt, tools)
    
    return {
          "answer": response['text'],
          "citations": chunks,
          "grounded": is_grounded(response['text'], chunks),
          "used_tool": response['tool_call'] != None
         }

In [87]:
await call_model('What is the NOI for Oak Hill Plaza in Q3-2026?', tools=tools)

ChatCompletionMessage(content="I'll find the property ID for Oak Hill Plaza and then calculate the NOI for Q3-2026.", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_ruu5uy3l', function=Function(arguments='{"name_query":"Oak Hill Plaza"}', name='find_property'), type='function', index=0)], reasoning='The user is asking for the NOI for Oak Hill Plaza in Q3-2026. Let me break down what I need to do:\n\n1. I need to find the property ID for "Oak Hill Plaza" using the find_property tool\n2. I need to check if the period is already in the normalized format - it is: "Q3-2026"\n3. Then I need to call calculate_noi with the property_id and "Q3-2026"\n\nStep 1: Find the property "Oak Hill Plaza"\nStep 2: Use the property ID from step 1\nStep 3: Calculate NOI for Q3-2026\n\nLet me start with find_property.')
{'name_query': 'Oak Hill Plaza'}
find_property
ChatCompletionMessage(content='', refusal=None, ro

{'text': 'The NOI for Oak Hill Plaza in Q3‑2026 is $154,000 [calculate_noi:tool].',
 'tool_call': {'name': 'calculate_noi',
  'arguments': {'property_id': 2, 'period': 'Q3-2026'}},
 'tool_calls': [{'name': 'find_property',
   'arguments': {'name_query': 'Oak Hill Plaza'}},
  {'name': 'calculate_noi',
   'arguments': {'property_id': 2, 'period': 'Q3-2026'}}]}

In [ ]:
"""
ChatCompletionMessage(content='Based on the available context, Maple Ridge Apartments’ total revenue in Q3 2026 was **$10,295,491** [3-2].'
, refusal=None, role='assistant'
, annotations=None, audio=None
, function_call=None, tool_calls=None
, reasoning="The user is asking for total revenue for Maple Ridge Apartments in Q3 2026. Let me examine the context chunks provided to see if this information is available.\n\nLooking at the available chunks [3-1] and [3-2]:\n- [3-1] defines Net Operating Income as Revenue minus operating expenses, and defines Operating Expenses as all ongoing and periodic costs except interest, taxes, capital expenditures, and depreciation. It also explicitly states that Revenue is total money received for goods or services.\n- [3-2] provides financial performance data for Maple Ridge Apartments in Q3-2026, including: Occupancy 95.8%, Revenue $10,295,491, Operating Expenses $6,659,799, Net Operating Income $3,635,692.\n\nSince the question asks about total revenue and not NOI, I should not use the calculate_noi tool. The context chunk [3-2] directly states the Revenue value for Maple Ridge Apartments in Q3-2026.\n\nThe answer should be: $10,295,491 [3-2].\n\nI don't need to call any tools for this question.")
"""